In [ ]:
lysosomal polygenic risk score analysis across GBA1-PD vs iPD/NMC/controls in GP2 Neurobooster genotyping data (all ancestries)

Project: GP2 lysosomal PRS

Version: Python/3.10.17, R/4.4.2

Notebook Overview

1. Description Loading Python libraries Set paths Make working directory

2. Installing packages

3. process clinical data; Create a covariate file with GP2 data

4.1 Run PRSice (GBAPD VS NMC IN EUR)
4.2 Run PRSice (GBAPD VS controls IN EUR)
4.3 Run PRSice (GBAPD VS iPD IN EUR)
5. plot ROC

6. GLM analysis adjusting for sex, age, PC1-5

Getting Started

Import python dependencies

In [1]:
## Import the necessary python dependencies 
%pip install seaborn --upgrade
!pip install rpy2
%load_ext rpy2.ipython
%pip install -U kaleido

from datetime import date
import importlib.metadata
from IPython.display import display
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, TwoSlopeNorm
from matplotlib.ticker import FormatStrFormatter
import matplotlib.gridspec as gridspec
import numbers
import numpy as np
import os
import pandas as pd
import plotly.express as px
import requests
import scipy
from scipy import stats
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
import subprocess
import sys
import seaborn as sns
import types
# Use pathlib for file path manipulation
import pathlib

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Define helper functions

In [2]:
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different imported names vs. system/pip names
        # Unfortunately, there is no systematic way to get pip names from a package's imported name. You'll have to add exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name

def min_max_scale(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

def compare_rocs(input_path, output_path, auc1_variable, auc2_variable, xlabel):
    # Read p-values matrix and rescale values
    df_pvals = pd.read_csv(input_path, sep="\t", index_col="Ancestry")
    df_pvals_log = np.log(-np.log(np.abs(df_pvals)) + 1) * df_pvals / np.abs(df_pvals)
    
    # Prepare formatting for heatmap
    max_abs = np.max(np.abs(df_pvals_log))
    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
    annot = df_pvals.map(lambda x: "*" if -0.05 <= x <= 0.05 else "")
    
    # Generate heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df_pvals_log, cmap="RdBu", norm=norm, cbar=True, ax=ax, annot=annot, fmt="")
    
    # Format color bar
    cbar = ax.collections[0].colorbar
    cbar.set_label("")
    cbar.set_ticks([-1.38522686, 1.38522686])
    cbar.set_ticklabels([f"{auc1_variable} Significantly Better", f"{auc2_variable} Significantly Better"])
    
    # Label axes and save figure
    plt.xlabel(xlabel)
    plt.ylabel("Target data ancestry")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

Print out versions of imported python dependencies

In [3]:
imports = list(set(get_imports()))
print(f"PACKAGE VERSIONS ({date})")
for m in importlib.metadata.distributions():
    if m.metadata["Name"] in imports and m.metadata["Name"]!="pip":
        print(f"\t{m.metadata['Name']}=={m.version}")

PACKAGE VERSIONS (21-MAY-2026)
	requests==2.32.3
	scipy==1.15.2
	seaborn==0.13.2
	plotly==6.0.0
	matplotlib==3.10.1
	statsmodels==0.14.4
	numpy==1.26.0
	pandas==2.2.1
	scikit-learn==1.6.1


Load R dependencies

In [4]:
%%R

install.packages("caret")
install.packages("optparse", repos="https://cloud.r-project.org/")

--- Please select a CRAN mirror for use in this session ---
Secure CRAN mirrors 

 1: 0-Cloud [https]                   2: Australia (Canberra) [https]   
 3: Australia (Melbourne 1) [https]   4: Australia (Melbourne 2) [https]
 5: Austria (Wien) [https]            6: Belgium (Brussels) [https]     
 7: Brazil (PR) [https]               8: Brazil (SP 1) [https]          
 9: Brazil (SP 2) [https]            10: Bulgaria [https]               
11: Canada (MB) [https]              12: Canada (ON 1) [https]          
13: Canada (ON 2) [https]            14: Chile (Santiago) [https]       
15: China (Beijing 1) [https]        16: China (Beijing 2) [https]      
17: China (Beijing 3) [https]        18: China (Hefei) [https]          
19: China (Hong Kong) [https]        20: China (Jinan) [https]          
21: China (Lanzhou) [https]          22: China (Nanjing) [https]        
23: China (Shanghai 2) [https]       24: China (Shenzhen) [https]       
25: China (Wuhan) [https]            26: C

Selection:  35


* installing *source* package ‘caret’ ...
** package ‘caret’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C compiler: ‘x86_64-conda-linux-gnu-cc (conda-forge gcc 14.2.0-2) 14.2.0’


x86_64-conda-linux-gnu-cc -I"/opt/conda/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /opt/conda/include -I/opt/conda/include -Wl,-rpath-link,/opt/conda/lib    -fpic  -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /opt/conda/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1734381570424/work=/usr/local/src/conda/r-base-4.4.2 -fdebug-prefix-map=/opt/conda=/usr/local/src/conda-prefix  -c caret.c -o caret.o
x86_64-conda-linux-gnu-cc -shared -L/opt/conda/lib/R/lib -Wl,-O2 -Wl,--sort-common -Wl,--as-needed -Wl,-z,relro -Wl,-z,now -Wl,--disable-new-dtags -Wl,--gc-sections -Wl,--allow-shlib-undefined -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib -o caret.so caret.o -L/opt/conda/lib/R/lib -lR


installing to /opt/conda/lib/R/library/00LOCK-caret/00new/caret/libs
** R
** data
** inst
** byte-compile and prepare package for lazy loading


Warning message:
package ‘lattice’ was built under R version 4.4.3 


** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location


** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location


** testing if installed package keeps a record of temporary installation path
* DONE (caret)
* installing *source* package ‘optparse’ ...
** package ‘optparse’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** exec
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (optparse)


trying URL 'https://ftp.fau.de/cran/src/contrib/caret_7.0-1.tar.gz'
Content type 'application/x-gzip' length 2273919 bytes (2.2 MB)
downloaded 2.2 MB


The downloaded source packages are in
	‘/tmp/RtmpRcfEn0/downloaded_packages’
Updating HTML index of packages in '.Library'
Making 'packages.html' ... done
trying URL 'https://cloud.r-project.org/src/contrib/optparse_1.8.2.tar.gz'
Content type 'application/x-gzip' length 49638 bytes (48 KB)
downloaded 48 KB


The downloaded source packages are in
	‘/tmp/RtmpRcfEn0/downloaded_packages’
Updating HTML index of packages in '.Library'
Making 'packages.html' ... done


In [5]:
%%R

require(data.table)
require(dplyr)
require(ggplot2)

library(optparse)
library(data.table)
library("ggplot2")
library(RColorBrewer)
library("caret")
library("pROC")
install.packages("ggplot2")
library(ggplot2)

* installing *source* package ‘ggplot2’ ...
** package ‘ggplot2’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (ggplot2)


Loading required package: data.table
data.table 1.17.4 using 24 threads (see ?getDTthreads).  Latest news: r-datatable.com
Loading required package: dplyr

Attaching package: ‘dplyr’

The following objects are masked from ‘package:data.table’:

    between, first, last

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Loading required package: ggplot2
Loading required package: lattice
Type 'citation("pROC")' for a citation.

Attaching package: ‘pROC’

The following objects are masked from ‘package:stats’:

    cov, smooth, var

trying URL 'https://ftp.fau.de/cran/src/contrib/ggplot2_4.0.3.tar.gz'
Content type 'application/x-gzip' length 6327703 bytes (6.0 MB)
downloaded 6.0 MB


The downloaded source packages are in
	‘/tmp/RtmpRcfEn0/downloaded_packages’
Updating HTML index of packages in '.Library'
Making 'packages.html' ... done
In addition: Warning messages:
1: pac

Define directories and ancestry lists

In [6]:
WORK_DIR = "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR = "/home/jupyter/workspace/ws_files/r11/results/aim2"
REL11_DIR = "/home/jupyter/workspace/gp2_tier2_eu_release11"

In [7]:
%%R

WORK_DIR <- "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR <- "/home/jupyter/workspace/ws_files/r11/results/aim2"
REL11_DIR <- "/home/jupyter/workspace/gp2_tier2_eu_release11"

Install bioinformatics packages

In [8]:
%%bash

if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter"
else
    echo "Plink is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
    unzip -o /home/jupyter/plink_linux_x86_64_20190304.zip -d /home/jupyter
    rm /home/jupyter/plink_linux_x86_64_20190304.zip
fi

chmod u+x /home/jupyter/plink

Plink is already installed in /home/jupyter


In [9]:
%%bash

if test -e /home/jupyter/plink2; then
    echo "Plink2 is already installed in /home/jupyter"
else
    echo "Plink2 is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
    unzip -o /home/jupyter/plink2_linux_x86_64_latest.zip -d /home/jupyter
    rm /home/jupyter/plink2_linux_x86_64_latest.zip
fi

chmod u+x /home/jupyter/plink2

Plink2 is already installed in /home/jupyter


In [10]:
%%bash

if test -e /home/jupyter/metal; then
    echo "Metal is already installed in /home/jupyter"
else
    echo "Metal is not installed"
    wget -P /home/jupyter https://csg.sph.umich.edu/abecasis/metal/download/Linux-metal.tar.gz
    tar --strip-components=1 -xzf /home/jupyter/Linux-metal.tar.gz -C /home/jupyter
    rm /home/jupyter/Linux-metal.tar.gz
fi

chmod u+x /home/jupyter/metal

Metal is already installed in /home/jupyter


In [11]:
%%bash

if test -e /home/jupyter/prsice; then
    echo "PRSice is already installed in /home/jupyter"
else
    echo "PRSice is not installed"
    wget -P /home/jupyter https://github.com/choishingwan/PRSice/releases/download/2.3.5/PRSice_linux.zip
    unzip -o /home/jupyter/PRSice_linux.zip -d /home/jupyter
    mv /home/jupyter/PRSice_linux /home/jupyter/prsice
fi

chmod u+x /home/jupyter/prsice

PRSice is already installed in /home/jupyter


Process Clinical Data

Generate covariates with PCs

In [8]:
CLINICAL_DATA_PATH = pathlib.Path(REL11_DIR, 'clinical_data/master_key_release11_final_vwb.csv')

In [ ]:
# Let's load the master key
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(key.shape)
key

In [ ]:
# Subsetting to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'nba_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE'}, inplace = True)
key

In [ ]:
# drop all NA values for all columns
key = key.dropna()
key

In [ ]:
no_mutations = pd.read_csv("/home/jupyter/workspace/ws_files/r11/results/covariate_IPD_rm.txt", sep='\s+')
no_mutations["Group"] = "no_mutations"
no_mutations = no_mutations[['FID', 'IID', 'Group', 'PHENO', 'SEX', 'AGE', 'ANCESTRY']]
no_mutations.rename(columns = {'PHENO':'phenotype',
                                     'ANCESTRY':'nba_label'}, inplace = True)
no_mutations

In [ ]:
no_mutations["phenotype"] = no_mutations["phenotype"].replace({
    1: "Control",
    2: "PD",
    -9: "Other"
})
no_mutations["SEX"] = no_mutations["SEX"].replace({
    1: "Male",
    2: "Female",
    0: "Other/Unknown/Not Reported"
})
no_mutations

In [ ]:
# filter out to only get eur
no_mutations = no_mutations[no_mutations["nba_label"].isin(["EUR"])]
no_mutations

In [15]:
no_mutations["phenotype"].value_counts(dropna=False)

phenotype
PD         25348
Other      14873
Control    10399
Name: count, dtype: int64

In [ ]:
# to filter out phenotype = 1 control IPD in EUR
no_mutations_controls_EUR = no_mutations[
    (no_mutations["phenotype"] == "Control")   # 1 = Control
]
no_mutations_controls_EUR

In [ ]:
# to filter out phenotype = 2 case IPD in EUR
IPD_EUR = no_mutations[
    (no_mutations["phenotype"] == "PD")   
]
IPD_EUR

In [ ]:
GBA_EUR = pd.read_csv("/home/jupyter/workspace/ws_files/r11/cohort/EUR/GBA1risk.samplestoKeep.rm.txt", sep='\t', header=None, names=['FID', 'IID'])
GBA_EUR = GBA_EUR.merge(key, on="IID", how="inner")
GBA_EUR

In [19]:
GBA_EUR["phenotype"].value_counts(dropna=False)

phenotype
PD         2433
Other      1095
Control     495
Name: count, dtype: int64

In [ ]:
# Reformat sex column
GBA_EUR['SEX'] = GBA_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
GBA_EUR["phenotype"] = GBA_EUR["phenotype"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(GBA_EUR)

In [ ]:
df_pcs = pd.read_csv(f'{REL11_DIR}/meta_data/qc_metrics/projected_pcs_vwb.csv')
df_pcs = df_pcs[['IID', 'PC1', 'PC2','PC3', 'PC4','PC5']].copy()
df_pcs['IID'] = df_pcs['IID'].str.replace(r'_s1$', '', regex=True)
df_pcs

In [ ]:
# merge GBAPD_EUR with pcs
IPD_EUR = IPD_EUR.merge(df_pcs, on="IID")
IPD_EUR

In [ ]:
# merge IPD_controls_EUR with pcs
no_mutations_controls_EUR = no_mutations_controls_EUR.merge(df_pcs, on="IID")
no_mutations_controls_EUR

In [ ]:
GBA_EUR = GBA_EUR.merge(df_pcs, on="IID")
GBA_EUR

In [ ]:
GBA_EUR = GBA_EUR[GBA_EUR["phenotype"] != -9].copy()
GBA_EUR

In [ ]:
# to filter GBA_EUR phenotype = 2
GBAPD_EUR = GBA_EUR[GBA_EUR["phenotype"] == 2].copy()
GBAPD_EUR

In [ ]:
# Rename columns to match desired output
IPD_EUR = IPD_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
IPD_EUR = IPD_EUR[columns_order]
IPD_EUR

In [ ]:
# Reformat sex column
IPD_EUR['SEX'] = IPD_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
IPD_EUR["PHENO"] = IPD_EUR["PHENO"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(IPD_EUR)

In [ ]:
# Rename columns to match desired output
GBAPD_EUR = GBAPD_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
GBAPD_EUR = GBAPD_EUR[columns_order]
GBAPD_EUR

In [ ]:
# Rename columns to match desired output
no_mutations_controls_EUR = no_mutations_controls_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
no_mutations_controls_EUR = no_mutations_controls_EUR[columns_order]
no_mutations_controls_EUR

In [ ]:
# Reformat sex column
no_mutations_controls_EUR['SEX'] = no_mutations_controls_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
no_mutations_controls_EUR["PHENO"] = no_mutations_controls_EUR["PHENO"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(no_mutations_controls_EUR)

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_HC = pd.concat([GBAPD_EUR, no_mutations_controls_EUR], ignore_index=True)
GBAPD_EUR_HC

In [34]:
GBAPD_EUR_HC["PHENO"].value_counts(dropna=False)

PHENO
1    10399
2     2391
Name: count, dtype: int64

In [35]:
GBAPD_EUR_HC_samplestokeep = GBAPD_EUR_HC[["FID", "IID"]].copy()
GBAPD_EUR_HC_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_samplestokeep.txt", index=False, sep="\t")

In [36]:
GBAPD_EUR_HC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_HC = GBAPD_EUR_HC[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_HC

In [38]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_HC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC.txt", sep='\t', index=False)

In [ ]:
# to filter GBA_EUR phenotype = 2
NMC = GBA_EUR[GBA_EUR["phenotype"] == 1].copy()
NMC

In [ ]:
# Rename columns to match desired output
NMC = NMC.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
NMC = NMC[columns_order]
NMC

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_NMC = pd.concat([GBAPD_EUR, NMC], ignore_index=True)
GBAPD_EUR_NMC

In [48]:
GBAPD_EUR_NMC["PHENO"].value_counts(dropna=False)

PHENO
2    2391
1     493
Name: count, dtype: int64

In [49]:
GBAPD_EUR_NMC_samplestokeep = GBAPD_EUR_NMC[["FID", "IID"]].copy()
GBAPD_EUR_NMC_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_samplestokeep.txt", index=False, sep="\t")

In [50]:
GBAPD_EUR_NMC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_NMC = GBAPD_EUR_NMC[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_NMC

In [52]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_NMC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC.txt", sep='\t', index=False)

In [ ]:
GBAPD_EUR

In [ ]:
IPD_EUR

In [55]:
# Make phenotype file for PRS: GBA1-PD cases vs iPD controls

# GBA1-PD cases
GBAPD_EUR_pheno = GBAPD_EUR[['FID', 'IID']].copy()
GBAPD_EUR_pheno['PHENO'] = 2

# iPD controls
IPD_EUR_pheno = IPD_EUR[['FID', 'IID']].copy()
IPD_EUR_pheno['PHENO'] = 1

# Combine cases and controls
GBAPD_EUR_IPD_PHENO = pd.concat(
    [GBAPD_EUR_pheno, IPD_EUR_pheno],
    ignore_index=True
)

# Check phenotype counts
GBAPD_EUR_IPD_PHENO['PHENO'].value_counts()

PHENO
1    25348
2     2391
Name: count, dtype: int64

In [56]:
GBAPD_EUR_IPD_PHENO.to_csv(
    "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt",
    sep="\t",
    index=False
)

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_IPD = pd.concat([GBAPD_EUR, IPD_EUR], ignore_index=True)
GBAPD_EUR_IPD

In [59]:
GBAPD_EUR_IPD["PHENO"].value_counts(dropna=False)

PHENO
2    27739
Name: count, dtype: int64

In [60]:
GBAPD_EUR_IPD_samplestokeep = GBAPD_EUR_IPD[["FID", "IID"]].copy()
GBAPD_EUR_IPD_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_samplestokeep.txt", index=False, sep="\t")

In [61]:
GBAPD_EUR_IPD.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_IPD = GBAPD_EUR_IPD[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_IPD

In [63]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_IPD.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD.txt", sep='\t', index=False)

Run PRSice (GBAPD VS HCcarriers IN EUR)

In [56]:
# GBAPD VS HCcarriers IN EUR
# the prevalence of GBA1-PD in GBA1 carriers
# cycle using SNP list without GBA and LRRK2
# remove --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--perm 10000 \
--bar-levels 0.1 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--no-full \
--fastscore \
--prevalence 0.05 \
--thread 16 \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC.txt

PRSice 2.3.5 (2021-09-20) 
https://github.com/choishingwan/PRSice
(C) 2016-2020 Shing Wan (Sam) Choi and Paul F. O'Reilly
GNU General Public License v3
If you use PRSice in any published work, please cite:
Choi SW, O'Reilly PF.
PRSice-2: Polygenic Risk Score Software for Biobank-Scale Data.
GigaScience 8, no. 7 (July 1, 2019)
2026-02-18 11:42:05
/home/jupyter/prsice \
    --a1 A1 \
    --a2 A2 \
    --bar-levels 0.001,0.05,0.1,0.2,0.3,0.4,0.5 \
    --base /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
    --beta  \
    --binary-target T \
    --bp BP \
    --chr CHR \
    --clump-kb 250kb \
    --clump-p 1.000000 \
    --clump-r2 0.100000 \
    --cov /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HCcarriers.txt \
    --fastscore  \
    --keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HCcarriers_samplestokeep.txt \
    --ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
    --no-full  \
    --num-auto 22

Estimate Specificity and Sensitivity

In [8]:
RESULTS_DIR

'/home/jupyter/workspace/ws_files/r11/results/aim2'

In [9]:
%%R

dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_HCcarriers_EUR/", "GBA_HCcarriers_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_NMC_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_HCcarriers_EUR/GBAPD_HCcarriers_EUR_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

   Ancestry      AUC  Accuracy  CI_Lower  CI_Upper Balanced_Accuracy
   <char>    <auc>     <num>     <num>     <num>             <num>
      EUR 0.620805 0.5483631 0.5293198 0.5673009         0.5926989
   Sensitivity Specificity
       <num>       <num>
   0.5235721   0.6618257


Setting levels: control = CONTROL, case = DISEASE
Setting direction: controls < cases


In [10]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBAPD_HCcarriers_EUR_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)


df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11.table_raw.txt", index=False, sep="\t")

/tmp/ipykernel_494/2078269956.py:27: RuntimeWarning: overflow encountered in scalar power
  lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
/tmp/ipykernel_494/2078269956.py:28: RuntimeWarning: overflow encountered in scalar power
  upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
/tmp/ipykernel_494/2078269956.py:29: RuntimeWarning: overflow encountered in scalar power
  df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"


Plot ROC

In [11]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/','/GBAPD_HCcarriers_EUR/', 'GBA_HCcarriers_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_NMC_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [12]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs NMC_EUR (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs NMC in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_ROC_raw.png", dpi=300)
plt.close()

regression analysis

In [13]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt")
head(data)

            FID          IID In_Regression           PRS   SEX   AGE ANCESTRY
       <char>       <char>        <char>         <num> <int> <num>   <char>
 ANDPD_000019 ANDPD_000019           Yes -5.953703e-05     1    65      EUR
 ANDPD_000020 ANDPD_000020           Yes  8.069391e-05     2    70      EUR
 ANDPD_000040 ANDPD_000040           Yes -1.674849e-04     1    77      EUR
  APGS_000019  APGS_000019           Yes -9.233868e-05     1    69      EUR
  APGS_000059  APGS_000059           Yes  1.089355e-04     2    64      EUR
  APGS_000061  APGS_000061           Yes  4.296847e-06     2    53      EUR
   PHENO       PC1       PC2        PC3       PC4      PC5  CASE probDisease
 <int>     <num>     <num>      <num>     <num>    <num> <int>       <num>
     2 -33.08147 -36.70553  -9.588107 -8.169611 21.48972     1   0.8156557
     2 -34.04302 -35.68237 -11.746944 -9.124783 21.90485     1   0.8625544
     2 -36.13275 -37.60417 -11.450701 -8.558405 21.35664     1   0.7717396
     2 -32.61

In [14]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.csv")

In [15]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

            FID          IID In_Regression           PRS   SEX   AGE ANCESTRY
       <char>       <char>        <char>         <num> <int> <num>   <char>
 ANDPD_000019 ANDPD_000019           Yes -5.953703e-05     1    65      EUR
 ANDPD_000020 ANDPD_000020           Yes  8.069391e-05     2    70      EUR
 ANDPD_000040 ANDPD_000040           Yes -1.674849e-04     1    77      EUR
  APGS_000019  APGS_000019           Yes -9.233868e-05     1    69      EUR
  APGS_000059  APGS_000059           Yes  1.089355e-04     2    64      EUR
  APGS_000061  APGS_000061           Yes  4.296847e-06     2    53      EUR
   PHENO       PC1       PC2        PC3       PC4      PC5  CASE probDisease
 <int>     <num>     <num>      <num>     <num>    <num> <int>       <num>
     2 -33.08147 -36.70553  -9.588107 -8.169611 21.48972     1   0.8156557
     2 -34.04302 -35.68237 -11.746944 -9.124783 21.90485     1   0.8625544
     2 -36.13275 -37.60417 -11.450701 -8.558405 21.35664     1   0.7717396
     2 -32.61

In [16]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [17]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

       Estimate   Std. Error    z value     Pr(>|z|)    Variable
   -0.45794600 1.150621e+00 -0.3979991 6.906309e-01 (Intercept)
 2659.38598199 3.037726e+02  8.7545282 2.049591e-18         PRS
   -0.66427415 1.040709e-01 -6.3829025 1.737626e-10         SEX
    0.02529409 4.604588e-03  5.4932363 3.946342e-08         AGE
   -0.02902292 2.935393e-02 -0.9887233 3.227985e-01         PC1
   -0.02251141 2.366458e-02 -0.9512704 3.414671e-01         PC2
   -0.01595043 2.537825e-02 -0.6285079 5.296713e-01         PC3
    0.01362355 2.545213e-02  0.5352617 5.924689e-01         PC4
   -0.01837114 1.084019e-02 -1.6947247 9.012766e-02         PC5


In [18]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

Run PRSice (GBAPD VS HC IN EUR)

In [19]:
# GBA VS HC IN EUR
# GBA-PD prevalence in the general population
# --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_EUR_r11_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--bar-levels 0.1 \
--fastscore \
--perm 10000 \
--prevalence 0.00005 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--no-full \
--thread 16 \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC.txt

PRSice 2.3.5 (2021-09-20) 
https://github.com/choishingwan/PRSice
(C) 2016-2020 Shing Wan (Sam) Choi and Paul F. O'Reilly
GNU General Public License v3
If you use PRSice in any published work, please cite:
Choi SW, O'Reilly PF.
PRSice-2: Polygenic Risk Score Software for Biobank-Scale Data.
GigaScience 8, no. 7 (July 1, 2019)
2025-08-13 12:13:12
/home/jupyter/prsice \
    --a1 A1 \
    --a2 A2 \
    --bar-levels 0.001,0.05,0.1,0.2,0.3,0.4,0.5 \
    --base /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
    --beta  \
    --binary-target T \
    --bp BP \
    --chr CHR \
    --clump-kb 250kb \
    --clump-p 1.000000 \
    --clump-r2 0.100000 \
    --cov /home/jupyter/workspace/ws_files/results/GBAPDvsHC_EUR.txt \
    --fastscore  \
    --keep /home/jupyter/workspace/ws_files/results/GBAPDvsHC_EUR.txt \
    --ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
    --no-full  \
    --num-auto 22 \
    --out /home/jupyter/workspace/ws_files/

Estimate Specificity and Sensitivity

In [19]:
RESULTS_DIR

'/home/jupyter/workspace/ws_files/r11/results/aim2'

In [20]:
%%R
dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_HC_EUR/", "GBA_HC_EUR_r11_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_HC_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_HC_EUR/GBA_HC_EUR_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

   Ancestry       AUC  Accuracy  CI_Lower  CI_Upper Balanced_Accuracy
   <char>     <auc>     <num>     <num>     <num>             <num>
      EUR 0.6207862 0.5768948 0.5680808 0.5856722         0.5863295
   Sensitivity Specificity
       <num>       <num>
   0.6010879   0.5715711


Setting levels: control = CONTROL, case = DISEASE
Setting direction: controls < cases


In [21]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)


df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11.table_raw.txt", index=False, sep="\t")

/tmp/ipykernel_494/3814150992.py:27: RuntimeWarning: overflow encountered in scalar power
  lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
/tmp/ipykernel_494/3814150992.py:28: RuntimeWarning: overflow encountered in scalar power
  upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
/tmp/ipykernel_494/3814150992.py:29: RuntimeWarning: overflow encountered in scalar power
  df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"


Plot ROC

In [22]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/GBAPD_HC_EUR/', 'GBA_HC_EUR_r11_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_HC_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [23]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs HC_EUR (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs HC in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_ROC_raw.png", dpi=300)
plt.close()

regression analysis

In [24]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt")
head(data)

             FID           IID In_Regression           PRS   SEX   AGE ANCESTRY
        <char>        <char>        <char>         <num> <int> <num>   <char>
 AAPDGC_000147 AAPDGC_000147           Yes  2.018298e-05     1    67      EUR
  ANDPD_000003  ANDPD_000003           Yes  1.935631e-04     1    30      EUR
  ANDPD_000011  ANDPD_000011           Yes -2.091152e-04     1    79      EUR
  ANDPD_000012  ANDPD_000012           Yes  3.958947e-04     2    61      EUR
  ANDPD_000016  ANDPD_000016           Yes -3.580750e-04     2    79      EUR
  ANDPD_000019  ANDPD_000019           Yes -6.049680e-05     1    65      EUR
   PHENO       PC1       PC2        PC3        PC4       PC5  CASE probDisease
 <int>     <num>     <num>      <num>      <num>     <num> <int>       <num>
     1 -14.65062 -30.33615  -7.811478 -15.787953 -8.395468     0  0.20640824
     1 -33.40639 -34.73791 -10.903264  -7.902955 23.873704     0  0.28551599
     1 -33.01238 -33.30622  -9.103843  -8.740768 24.630493     0

In [25]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_r11_risk_results_raw.csv")

In [26]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

             FID           IID In_Regression           PRS   SEX   AGE ANCESTRY
        <char>        <char>        <char>         <num> <int> <num>   <char>
 AAPDGC_000147 AAPDGC_000147           Yes  2.018298e-05     1    67      EUR
  ANDPD_000003  ANDPD_000003           Yes  1.935631e-04     1    30      EUR
  ANDPD_000011  ANDPD_000011           Yes -2.091152e-04     1    79      EUR
  ANDPD_000012  ANDPD_000012           Yes  3.958947e-04     2    61      EUR
  ANDPD_000016  ANDPD_000016           Yes -3.580750e-04     2    79      EUR
  ANDPD_000019  ANDPD_000019           Yes -6.049680e-05     1    65      EUR
   PHENO       PC1       PC2        PC3        PC4       PC5  CASE probDisease
 <int>     <num>     <num>      <num>      <num>     <num> <int>       <num>
     1 -14.65062 -30.33615  -7.811478 -15.787953 -8.395468     0  0.20640824
     1 -33.40639 -34.73791 -10.903264  -7.902955 23.873704     0  0.28551599
     1 -33.01238 -33.30622  -9.103843  -8.740768 24.630493     0

In [27]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [28]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

       Estimate   Std. Error     z value     Pr(>|z|)    Variable
 -3.346583e+00 5.272288e-01  -6.3474972 2.188459e-10 (Intercept)
  2.481137e+03 1.394345e+02  17.7942801 7.826669e-71         PRS
 -5.458405e-01 4.901172e-02 -11.1369390 8.292058e-29         SEX
  1.747074e-02 2.051078e-03   8.5178353 1.625632e-17         AGE
 -3.833994e-02 1.360819e-02  -2.8174164 4.841172e-03         PC1
 -2.760557e-03 1.079588e-02  -0.2557047 7.981788e-01         PC2
 -4.486845e-02 1.095578e-02  -4.0954121 4.214180e-05         PC3
  3.891121e-02 1.189194e-02   3.2720664 1.067645e-03         PC4
  4.614620e-03 4.929433e-03   0.9361361 3.492031e-01         PC5


In [29]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

Run PRSice (GBAPD VS IPD IN EUR)

In [ ]:
# GBAPD VS IPD IN EUR
# GBA1-PD prevalence in PD
# cycle using SNP list without GBA and LRRK2 v1 with prevalence
# --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--perm 10000 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--fastscore \
--no-full \
--thread 16 \
--prevalence 0.1 \
--bar-levels 0.1 \
--pheno-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt \
--pheno-col PHENO \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD.txt

In [86]:
! head /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt

FID	IID	PHENO
ANDPD_000019	ANDPD_000019	2
ANDPD_000020	ANDPD_000020	2
ANDPD_000040	ANDPD_000040	2
APGS_000019	APGS_000019	2
APGS_000059	APGS_000059	2
APGS_000061	APGS_000061	2
APGS_000063	APGS_000063	2
APGS_000076	APGS_000076	2
APGS_000084	APGS_000084	2


Estimate Specificity and Sensitivity

In [30]:
RESULTS_DIR

'/home/jupyter/workspace/ws_files/r11/results/aim2'

In [92]:
%%R

head(dat)

           IID        FID.x In_Regression        PRS        FID.y SEX AGE
 ANDPD_000002 ANDPD_000002           Yes  0.9578283 ANDPD_000002   1  65
 ANDPD_000005 ANDPD_000005           Yes  0.1233798 ANDPD_000005   1  62
 ANDPD_000006 ANDPD_000006           Yes  0.1359992 ANDPD_000006   1  57
 ANDPD_000008 ANDPD_000008           Yes  0.3979907 ANDPD_000008   2  74
 ANDPD_000014 ANDPD_000014           Yes -2.0835244 ANDPD_000014   1  34
 ANDPD_000015 ANDPD_000015           Yes -0.4567407 ANDPD_000015   1  60
  ANCESTRY PHENO       PC1       PC2       PC3        PC4      PC5 CASE zSCORE
      EUR     2 -34.02813 -35.33816 -13.83362  -9.200535 23.66819    1    NaN
      EUR     2 -33.87669 -34.76137 -13.63779  -7.593147 22.11362    1    NaN
      EUR     2 -35.92408 -33.55596 -14.03525  -8.571577 27.18162    1    NaN
      EUR     2 -33.39685 -38.03571 -12.51555 -11.110218 19.90177    1    NaN
      EUR     2 -30.70452 -34.38192 -13.70385  -6.783292 17.70767    1    NaN
      EUR     2 -30

In [93]:
%%R

head(cov)

           FID          IID SEX AGE ANCESTRY PHENO       PC1       PC2
 ANDPD_000019 ANDPD_000019   1  65      EUR     2 -33.08147 -36.70553
 ANDPD_000020 ANDPD_000020   2  70      EUR     2 -34.04302 -35.68237
 ANDPD_000040 ANDPD_000040   1  77      EUR     2 -36.13275 -37.60417
  APGS_000019  APGS_000019   1  69      EUR     2 -32.61688 -34.90407
  APGS_000059  APGS_000059   2  64      EUR     2 -33.47918 -36.07158
  APGS_000061  APGS_000061   2  53      EUR     2 -32.43685 -35.46908
         PC3       PC4      PC5
  -9.588107 -8.169611 21.48972
 -11.746944 -9.124783 21.90485
 -11.450701 -8.558405 21.35664
 -12.540508 -7.876638 22.87913
 -11.141049 -8.657989 26.64496
 -11.500045 -7.736682 25.99578


In [96]:
# GBA1-PD cases
GBAPD_EUR_case = GBAPD_EUR.copy()
GBAPD_EUR_case["PHENO"] = 2

# iPD controls
IPD_EUR_control = IPD_EUR.copy()
IPD_EUR_control["PHENO"] = 1

# combine
GBAPD_EUR_IPD_covariate = pd.concat(
    [GBAPD_EUR_case, IPD_EUR_control],
    ignore_index=True
)

# check
GBAPD_EUR_IPD_covariate["PHENO"].value_counts()

PHENO
1    25348
2     2391
Name: count, dtype: int64

In [97]:
GBAPD_EUR_IPD_covariate.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_covariate.txt", index=False, sep="\t")

In [31]:
%%R

dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_IPD/", "GBAPD_IPD_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_IPD_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_IPD/GBAPD_IPD_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

   Ancestry       AUC  Accuracy  CI_Lower  CI_Upper Balanced_Accuracy
   <char>     <auc>     <num>     <num>     <num>             <num>
      EUR 0.5150914 0.5407133 0.5345915 0.5468259         0.5134425
   Sensitivity Specificity
       <num>       <num>
   0.4805077   0.5463772


Setting levels: control = CONTROL, case = DISEASE
Setting direction: controls < cases


In [32]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)

df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11.table_raw.txt", index=False, sep="\t")

Plot ROC

In [33]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/','/GBAPD_IPD/', 'GBAPD_IPD_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_IPD_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [34]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs iPD (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs iPD in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_ROC_raw.png", dpi=300)
plt.close()

regression  analysis

In [35]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt")
head(data)

            FID          IID In_Regression           PRS   SEX   AGE ANCESTRY
       <char>       <char>        <char>         <num> <int> <num>   <char>
 ANDPD_000002 ANDPD_000002           Yes  1.484927e-04     1    65      EUR
 ANDPD_000005 ANDPD_000005           Yes  1.890766e-06     1    62      EUR
 ANDPD_000006 ANDPD_000006           Yes  4.107836e-06     1    57      EUR
 ANDPD_000008 ANDPD_000008           Yes  5.013639e-05     2    74      EUR
 ANDPD_000014 ANDPD_000014           Yes -3.858342e-04     1    34      EUR
 ANDPD_000015 ANDPD_000015           Yes -1.000290e-04     1    60      EUR
   PHENO       PC1       PC2       PC3        PC4      PC5  CASE probDisease
 <int>     <num>     <num>     <num>      <num>    <num> <int>       <num>
     1 -34.02813 -35.33816 -13.83362  -9.200535 23.66819     0  0.09029997
     1 -33.87669 -34.76137 -13.63779  -7.593147 22.11362     0  0.08637824
     1 -35.92408 -33.55596 -14.03525  -8.571577 27.18162     0  0.08643638
     1 -33.39

In [36]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.csv")

In [37]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

            FID          IID In_Regression           PRS   SEX   AGE ANCESTRY
       <char>       <char>        <char>         <num> <int> <num>   <char>
 ANDPD_000002 ANDPD_000002           Yes  1.484927e-04     1    65      EUR
 ANDPD_000005 ANDPD_000005           Yes  1.890766e-06     1    62      EUR
 ANDPD_000006 ANDPD_000006           Yes  4.107836e-06     1    57      EUR
 ANDPD_000008 ANDPD_000008           Yes  5.013639e-05     2    74      EUR
 ANDPD_000014 ANDPD_000014           Yes -3.858342e-04     1    34      EUR
 ANDPD_000015 ANDPD_000015           Yes -1.000290e-04     1    60      EUR
   PHENO       PC1       PC2       PC3        PC4      PC5  CASE probDisease
 <int>     <num>     <num>     <num>      <num>    <num> <int>       <num>
     1 -34.02813 -35.33816 -13.83362  -9.200535 23.66819     0  0.09029997
     1 -33.87669 -34.76137 -13.63779  -7.593147 22.11362     0  0.08637824
     1 -35.92408 -33.55596 -14.03525  -8.571577 27.18162     0  0.08643638
     1 -33.39

In [38]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [39]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

       Estimate   Std. Error    z value     Pr(>|z|)    Variable
  -2.192999146 4.928229e-01 -4.4498723 8.592135e-06 (Intercept)
 281.077073721 1.266757e+02  2.2188715 2.649546e-02         PRS
   0.135816377 4.564143e-02  2.9757259 2.922960e-03         SEX
  -0.020397622 2.095071e-03 -9.7360043 2.117159e-22         AGE
  -0.008218386 1.230928e-02 -0.6676575 5.043522e-01         PC1
  -0.011080361 1.028395e-02 -1.0774426 2.812826e-01         PC2
  -0.024921684 1.071103e-02 -2.3267313 1.997957e-02         PC3
   0.019477999 1.100376e-02  1.7701219 7.670683e-02         PC4
   0.010861845 4.563447e-03  2.3801844 1.730398e-02         PC5


In [40]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

In [41]:
# visualization
# make ROC curves together in one plot

def plot_roc(ax, filepath, label, y="CASE", score="probDisease", lw=2):
    df = pd.read_csv(filepath, sep="\t")
    fpr, tpr, _ = roc_curve(df[y], df[score])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=lw, label=f"{label} (AUC = {roc_auc:.2f})")
    return roc_auc

# --- one plot ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.set_palette("Dark2")  # optional styling

# diagonal
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")

# add each ROC curve
plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt",
    label="GBA1-PD vs NMC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt",
    label="GBA1-PD vs HC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt",
    label="GBA1-PD vs IPD(EUR)"
)

ax.set_title("ROC Curves: GBA1-PD vs HC,NMC,and iPD(EUR)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(f"{RESULTS_DIR}/GBA1_r11_ROC_raw.png", dpi=300)
plt.close(fig)

In [108]:
# visualization
# make ROC curves together in one plot

def plot_roc(ax, filepath, label, y="CASE", score="probDisease", lw=2):
    df = pd.read_csv(filepath, sep="\t")
    fpr, tpr, _ = roc_curve(df[y], df[score])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=lw, label=f"{label} (AUC = {roc_auc:.2f})")
    return roc_auc

# --- one plot ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.set_palette("Dark2")  # optional styling

# diagonal
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")

# add each ROC curve
plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_std.txt",
    label="GBA1-PD vs NMC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_std.txt",
    label="GBA1-PD vs HC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_std.txt",
    label="GBA1-PD vs IPD(EUR)"
)

ax.set_title("ROC Curves: GBA1-PD vs HC,NMC,and iPD(EUR)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(f"{RESULTS_DIR}/GBA1_r11_ROC_std.png", dpi=300)
plt.close(fig)